# Data quality measurement

Writes the response table and the by-dimension figure from the baseline sweep.

In [1]:
from itertools import groupby
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from figures import (
    COLORS,
    LEGEND_SIZE,
    PAPER_STYLE,
    SCENARIO_DASHES,
    TEXT_WIDTH_PT,
    as_percent,
    as_tex,
    measure_label,
    scenario_label,
    set_size,
)
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator

from experiments.measures import measure_dimension

RESULTS = Path("../measurements/data-quality-measurement")
OUT = Path("../results")

raw = pd.concat(pd.read_parquet(p) for p in sorted(RESULTS.glob("**/results.parquet")))
obs = raw[raw.status == "ok"].copy()
obs["column"] = obs["column"].fillna("(table)")
obs = obs.astype({"dq_score": float, "error_rate": float, "column_error_rate": float})
print(f"{len(raw):,} rows · {raw.scenario.nunique()} scenarios · {raw.measure.nunique()} measures")

22,165 rows · 7 scenarios · 15 measures


In [2]:
# The change in score, column by column: the same measure, column and fold, with errors minus without.
# Only the columns a scenario really corrupted count.
key = ["measure", "column", "fold"]
clean = obs[obs.scenario == "clean"].set_index(key).dq_score
damaged = obs[(obs.scenario != "clean") & (obs.column_error_rate > 0)]
delta = damaged.assign(
    change=damaged.dq_score.to_numpy() - clean.reindex(pd.MultiIndex.from_frame(damaged[key])).to_numpy()
)

## Response table

A cell is the mean change in score from clean data to a 50% error rate, over the columns the scenario corrupted,
± the standard deviation over the five folds. Bold marks a change of at least 0.05, a dash a measure that applies
to none of the corrupted columns. The `clean` column is the measure's clean score over all columns.

In [3]:
TABLE_SCENARIOS = ["missing", "blank_record", "typo", "swap", "outlier", "wrong_unit"]
RELEVANT = 0.05

worst = delta[delta.error_rate == delta.error_rate.max()]
by_fold = worst.groupby(["measure", "scenario", "fold"]).change.mean()
# A measure that scores NaN on the corrupted columns has no cell and prints as a dash.
table = by_fold.groupby(["measure", "scenario"]).agg(mean="mean", sd="std").dropna(subset=["mean"])
baselines = obs[obs.scenario == "clean"].groupby("measure").dq_score.mean()


def cell_tex(measure, scenario):
    """The mean change, bold if relevant, and the fold SD beside it; a dash where the measure did not run."""
    if (measure, scenario) not in table.index:
        return "--"
    mean, sd = table.loc[(measure, scenario)]
    body = f"{mean:+.3f}"
    body = rf"\textbf{{{body}}}" if abs(mean) >= RELEVANT else body
    return body if pd.isna(sd) else rf"{body}{{\scriptsize$\pm{sd:.3f}$}}"


lines = [
    r"\begin{table*}[t]",
    r"  \centering",
    r"  \footnotesize",
    r"  \setlength{\tabcolsep}{2pt}",
    r"  \caption{%",
    r"    Average change in DQM scores from clean data to a 50\% per-column error rate.",
    r"    Each error scenario corrupts multiple columns, and we report the average.",
    r"    Changes of at least $0.05$ are marked bold, and a dash indicates that a DQM isn't "
    r"compatible with an error scenario.",
    r"    Each mean carries the standard deviation of the five folds beside it; a mean printed alone "
    r"was measured on one fold only.",
    r"  }",
    r"  \label{tab:overview-data-quality}",
    r"  \begin{tabular}{lrrrrrrr}",
    r"    \toprule",
    r"    & & \multicolumn{6}{c}{Error Scenario} \\",
    r"    \cmidrule(lr){3-8}",
    "    Data Quality Measure & clean & "
    + " & ".join(rf"\texttt{{{scenario_label(s)}}}" for s in TABLE_SCENARIOS)
    + r" \\",
    r"    \midrule",
]
# Grouped by ISO dimension, alphabetical within each group.
measures = sorted(table.index.unique("measure"), key=lambda m: (measure_dimension(m), measure_label(m)))
for index, (_dimension, group) in enumerate(groupby(measures, key=measure_dimension)):
    if index:
        lines.append(r"    \addlinespace")
    for measure in group:
        cells = " & ".join(cell_tex(measure, s) for s in TABLE_SCENARIOS)
        lines.append(f"    {measure_label(measure)} & {baselines[measure]:.3f} & {cells} \\\\")
lines += [r"    \bottomrule", r"  \end{tabular}", r"\end{table*}"]

(OUT / "response_table.tex").write_text("\n".join(lines) + "\n")
table.reset_index().to_csv(OUT / "response_table.csv", index=False)
print((OUT / "response_table.tex").read_text())

\begin{table*}[t]
  \centering
  \footnotesize
  \setlength{\tabcolsep}{2pt}
  \caption{%
    Average change in DQM scores from clean data to a 50\% per-column error rate.
    Each error scenario corrupts multiple columns, and we report the average.
    Changes of at least $0.05$ are marked bold, and a dash indicates that a DQM isn't compatible with an error scenario.
    Each mean carries the standard deviation of the five folds beside it; a mean printed alone was measured on one fold only.
  }
  \label{tab:overview-data-quality}
  \begin{tabular}{lrrrrrrr}
    \toprule
    & & \multicolumn{6}{c}{Error Scenario} \\
    \cmidrule(lr){3-8}
    Data Quality Measure & clean & \texttt{missing} & \texttt{blank record} & \texttt{typo} & \texttt{swap} & \texttt{outlier} & \texttt{wrong unit} \\
    \midrule
    Data Accuracy Range & 0.999 & +0.001{\scriptsize$\pm0.001$} & +0.001{\scriptsize$\pm0.001$} & -- & -- & \textbf{-0.348}{\scriptsize$\pm0.013$} & +0.000{\scriptsize$\pm0.000$} \\
    Ri

## Change in score, by dimension

One line per measure and scenario: the mean change in score from clean data over the corrupted columns, one panel
per ISO dimension, each scaled to its own range. `RiskOfDataInconsistency` is left out: it is the only measure
whose score rises under corruption, and it would take up much of the y range.

In [4]:
EXCLUDED = ["RiskOfDataInconsistency"]
DIMENSIONS = ["Accuracy", "Completeness", "Consistency"]
FIGURE = "2026-09-17-data-quality-measurement-by-dimension.pdf"
# The lines the paper discusses: label, measure, leader line targets, label position.
LABELS = {
    "Accuracy": [
        ("Semantic\nData Accuracy", "SemanticDataAccuracy", [("swap", 0.22)], (0.03, -0.45)),
        ("Risk of Data Set\nInaccuracy", "RiskOfDataSetInaccuracy", [("wrong_unit", 0.42)], (0.29, -0.055)),
    ],
    "Completeness": [
        ("Record\nCompleteness", "RecordCompleteness", [("missing", 0.07)], (0.12, -0.45)),
        ("Value Occurrence\nCompleteness", "ValueOccurrenceCompleteness", [("swap", 0.44)], (0.29, -0.68)),
    ],
    "Consistency": [
        ("Semantic\nConsistency", "SemanticConsistency", [("swap", 0.33), ("typo", 0.36)], (0.31, -0.55)),
    ],
}

lines = delta[~delta.measure.isin(EXCLUDED)].groupby(["measure", "scenario", "error_rate"]).change.mean()
lines = lines.dropna().reset_index()
lines["dimension"] = lines.measure.map(measure_dimension)
rates = [0.0, *sorted(lines.error_rate.unique())]
labelled = np.arange(0, max(rates) + 1e-9, 0.1)


def on_line(measure, scenario, rate):
    """The point at `rate` on a line."""
    line = lines[(lines.measure == measure) & (lines.scenario == scenario)]
    return rate, np.interp(rate, [0.0, *line.error_rate], [0.0, *line.change])


with plt.rc_context(PAPER_STYLE):
    fig, axes = plt.subplots(
        1,
        len(DIMENSIONS),
        figsize=set_size(TEXT_WIDTH_PT, subplots=(1, len(DIMENSIONS)), height_ratio=1.6),
        sharex=True,
        layout="constrained",
    )
    fig.get_layout_engine().set(w_pad=1 / 72, h_pad=1 / 72, wspace=0.04)
    for ax, dimension in zip(axes, DIMENSIONS, strict=True):
        ax.axhline(0.0, color="#777777", lw=0.7, zorder=1.5)
        for (_measure, scenario), line in lines[lines.dimension == dimension].groupby(["measure", "scenario"]):
            ax.plot(
                [0.0, *line.error_rate],
                [0.0, *line.change],
                color=COLORS[scenario],
                lw=1.15,
                alpha=0.95,
                linestyle=SCENARIO_DASHES[scenario],
                solid_capstyle="round",
                dash_capstyle="round",
                zorder=3,
            )
        for text, measure, targets, at in LABELS[dimension]:
            for index, (scenario, rate) in enumerate(targets):
                ax.annotate(
                    text,
                    xy=on_line(measure, scenario, rate),
                    xytext=at,
                    fontsize=LEGEND_SIZE,
                    color="black" if index == 0 else "none",
                    linespacing=1.0,
                    va="center",
                    bbox={"boxstyle": "square,pad=0.15", "fc": "none", "ec": "none"},
                    arrowprops={"arrowstyle": "-", "color": "#555555", "lw": 0.5, "shrinkA": 2, "shrinkB": 1},
                    zorder=4,
                )
        ax.set_xticks(labelled, [as_percent(rate) for rate in labelled])
        ax.set_xticks(rates, minor=True)
        ax.set_xlim(-0.03 * max(rates), 1.03 * max(rates))
        ax.yaxis.set_major_locator(MaxNLocator(nbins=5, steps=[1, 2, 2.5, 5, 10]))
        ax.set_title(dimension)
    axes[1].set_xlabel("Share of Erroneous Cells")
    fig.supylabel("Change in DQ Score")
    keys = [
        Line2D([], [], color=COLORS[s], lw=1.2, linestyle=SCENARIO_DASHES[s], label=as_tex(scenario_label(s)))
        for s in sorted(set(lines.scenario))
    ]
    fig.legend(handles=keys, loc="outside lower center", ncol=len(keys), columnspacing=1.0)
    fig.savefig(OUT / FIGURE)
    plt.close(fig)